# Qwen2.5-7B-Instruct zero-shot baseline

**Before running:** In Colab, go to `Runtime > Change runtime type` and pick a GPU (A100 or T4).

This notebook needs two files from your `insurance-claims-extraction` folder:
- `score_utils.py`
- `artifacts/dataset_full.json`

Cell 2 below will prompt you to upload them.

In [5]:
!pip install -q transformers accelerate bitsandbytes torch

In [6]:
import os
os.makedirs('artifacts', exist_ok=True)

from google.colab import files

print('Upload score_utils.py')
uploaded = files.upload()

print('Upload dataset_full.json (from your artifacts/ folder)')
uploaded2 = files.upload()
for fname in uploaded2:
    os.rename(fname, 'artifacts/dataset_full.json')

Upload score_utils.py


Saving score_utils.py to score_utils (1).py
Upload dataset_full.json (from your artifacts/ folder)


Saving dataset_full.json to dataset_full.json


In [4]:
import json
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

SYSTEM_PROMPT = """You extract structured data from insurance claim narratives.
Given a claim text, output ONLY a JSON object matching this shape (use null for
anything not present in the text -- do not guess or fabricate values):

{
  "header": {"claim_id": ..., "report_date": ..., "incident_date": ..., "reported_by": ..., "channel": ...},
  "policy_details": {"policy_number": ..., "policyholder_name": ..., "coverage_type": ..., "effective_date": ..., "expiration_date": ...} or null,
  "insured_objects": [...] or null,
  "incident_description": {"incident_type": ..., "location_type": ..., "estimated_damage_amount": ...} or null
}

Output raw JSON only, no markdown fences, no commentary."""

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb_config, device_map="auto"
)

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/27.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [7]:
def extract(claim_text: str) -> dict:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": claim_text},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    start = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=512, do_sample=False, temperature=None, top_p=None
        )
    latency = time.time() - start

    gen_tokens = out[0][inputs["input_ids"].shape[1]:]
    raw = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

    try:
        parsed = json.loads(raw)
    except json.JSONDecodeError:
        parsed = {}

    return {
        "prediction": parsed,
        "latency_s": round(latency, 3),
        "completion_tokens": len(gen_tokens),
        "raw_valid_json": bool(parsed),
        "raw_output": raw,
    }

In [8]:
with open("artifacts/dataset_full.json") as f:
    examples = json.load(f)

results = []
for i, ex in enumerate(examples):
    print(f"[{i+1}/{len(examples)}] extracting...")
    r = extract(ex["claim_text"])
    r["ground_truth"] = ex["ground_truth"]
    results.append(r)

with open("artifacts/qwen_baseline_results.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

valid_rate = sum(r["raw_valid_json"] for r in results) / len(results)
avg_latency = sum(r["latency_s"] for r in results) / len(results)
print(f"JSON validity rate: {valid_rate:.2%}")
print(f"Avg latency/request: {avg_latency:.2f}s")

[1/30] extracting...
[2/30] extracting...
[3/30] extracting...
[4/30] extracting...
[5/30] extracting...
[6/30] extracting...
[7/30] extracting...
[8/30] extracting...
[9/30] extracting...
[10/30] extracting...
[11/30] extracting...
[12/30] extracting...
[13/30] extracting...
[14/30] extracting...
[15/30] extracting...
[16/30] extracting...
[17/30] extracting...
[18/30] extracting...
[19/30] extracting...
[20/30] extracting...
[21/30] extracting...
[22/30] extracting...
[23/30] extracting...
[24/30] extracting...
[25/30] extracting...
[26/30] extracting...
[27/30] extracting...
[28/30] extracting...
[29/30] extracting...
[30/30] extracting...
JSON validity rate: 96.67%
Avg latency/request: 27.91s


In [9]:
from google.colab import files
files.download('artifacts/qwen_baseline_results.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>